[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/skarma91/gen-ai-and-agentic-ai-course/blob/main/modules/module-1-python-for-ai/milestone-assignment/assignment.ipynb)

# Module 1 milestone: data facts, an LLM summary, saved to JSON

Put the whole module together in one small tool. It should:

1. **Load** a dataset (the CSV below) into a Pandas DataFrame.
2. **Compute** a few facts from it with Pandas (a type-hinted function returning a dict).
3. **Build a prompt** string from those facts.
4. **Ask an LLM** to summarize them (local Ollama; wrap the call in `try`/`except` so a missing service does not crash the tool).
5. **Save** the facts plus the summary to a JSON file, using a small class.

**Deliverable:** the working notebook and the JSON it writes. The LLM step needs Ollama running locally; without it, your `try`/`except` should fall back to a clear message so the rest still runs.

**Skills exercised:** functions, type hints, a class, files and JSON, Pandas, and an LLM call, everything from Module 1.

### The dataset

In [1]:
csv_text = """product,category,units,revenue
Widget,hardware,120,2400
Gadget,hardware,80,3200
Cable,accessory,300,1500
Case,accessory,150,1200
App,software,50,5000"""

import pandas as pd
from io import StringIO
df = pd.read_csv(StringIO(csv_text))
print(df)

  product   category  units  revenue
0  Widget   hardware    120     2400
1  Gadget   hardware     80     3200
2   Cable  accessory    300     1500
3    Case  accessory    150     1200
4     App   software     50     5000


### 1. Compute facts (Pandas)

In [13]:
# TODO: return a dict of a few facts about df, for example:
#   rows, total_revenue, top_category_by_revenue, avg_units
# Use groupby / sum / mean / idxmax as needed. Give it a type hint.
def compute_facts(df: pd.DataFrame) -> dict:
    ...  # your code here
    facts_dict = {}
    facts_dict['rows'] = len(df)
    facts_dict['total_revenue'] = int(df['revenue'].sum())
    facts_dict['top_category_by_revenue'] = df.groupby('category')['revenue'].sum().idxmax()
    facts_dict['avg_units'] = float(df['units'].mean())
    return facts_dict

facts = compute_facts(df)
print(facts)

{'rows': 5, 'total_revenue': 13300, 'top_category_by_revenue': 'hardware', 'avg_units': 140.0}


### 2. Build a prompt from the facts

In [14]:
import json
# TODO: return a prompt string that asks for a short summary and includes the facts.
def build_prompt(facts: dict) -> str:
    prompt=f'''You are an analyst.Summarize the facts from the given dictionary{facts} in 2-3 sentences'''
    return prompt

### 3. Ask the LLM (with a graceful fallback)

In [15]:
import requests
# TODO: POST the prompt to Ollama at http://localhost:11434/api/generate and return
# resp.json()["response"]. Wrap it in try/except so that if Ollama is not running you
# return a clear fallback message instead of crashing.
def ask_llm(prompt: str) -> str:
    try:
        res = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "llama3.2",
            "prompt": prompt,
            "stream": False,
        },)
        return res.json()["response"]
    except requests.exceptions.ConnectionError:
       return "Could not reach Ollama — is it running? Returning fallback summary."

### 4. Save facts + summary to JSON

In [16]:
# TODO: a small class that saves {"facts": ..., "summary": ...} to a JSON file.
class Report:
    def __init__(self, path: str):
        self.path = path

    def save(self, facts: dict, summary: str) -> None:
        data = {"facts": facts, "summary": summary}
        with open(self.path, "w") as f:
            json.dump(data, f, indent=2)


# Put it together:
facts = compute_facts(df)
summary = ask_llm(build_prompt(facts))
Report("report.json").save(facts, summary)